# llm_sched

> Scheduling live LLM workloads (speedup plan Part A): longest runs first in one process pool, a cap check with a margin for runs and calls in flight (falling back to grid-by-grid priority order), refused runs recorded, rows grouped by grid whatever the mode, and per-call timing merged and summarised.

In [ ]:
#| default_exp llm_sched

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import json
from functools import partial
from pathlib import Path

import numpy as np

from explRL.bench_grid import rowKey, runGrid
from explRL.llm_budget import TOKEN_PROFILE, Budget, BudgetExceeded, maxCallCost
from explRL.llm_grid import estimateGrid, runAnyCell, withClient
from explRL.llm_runs import LLMConfig


def cellKey(c):
    "The keys a cell shares with its rows (study, factor, level, channel, N, seed)."
    return (str(c['study']), str(c['factor']), str(c['level']), str(c['cfg']['channel']), int(c['N']), int(c['seed']))


def _rowCell(r):
    return (str(r['study']), str(r['factor']), str(r['level']), str(r['channel']), int(float(r['N'])), int(float(r['seed'])))


def seqCalls(cell):
    "Sequential calls of one LLM run: LLM agents x turns per agent x episodes, plus one belief call per agent per episode."
    if cell['runner'] != 'llm':
        return 0
    cfg = LLMConfig.fromDict(cell['cfg'])
    return (cfg.N - cfg.nLiar) * cfg.episodes * (cfg.turnsPerAgent() + (cfg.task == 'forage'))


def orderCells(cells, latency=None):
    """Longest expected duration first: sequential calls x seconds per call for (model, task) (`latency` from a timing
    summary, else proportional to the profile's output tokens); numpy cells last; ties by cell key."""
    def dur(c):
        if c['runner'] != 'llm':
            return -1.0
        k = (c['cfg']['model'], c['cfg']['task'])
        return seqCalls(c) * ((latency or {}).get(k) or TOKEN_PROFILE.get(k, (0, 0, 500))[2])
    return sorted(cells, key=lambda c: (-dur(c), cellKey(c)))


def _maxCall(c):
    return maxCallCost(c['cfg']['model'], c['cfg']['task'], c['cfg']['maxTokens'])


def _runCost(c):
    return sum(v['cost'] for v in estimateGrid([c]).values())


def checkWorkload(cells, budget, workers):
    "-> (fits, need, left): need = run estimates + workers x the largest run estimate + workers x the largest call cost."
    llm = [c for c in cells if c['runner'] == 'llm']
    left = budget.remaining()
    if not llm:
        return True, 0.0, left
    est = [_runCost(c) for c in llm]
    need = sum(est) + workers * max(est) + workers * max(_maxCall(c) for c in llm)
    return need <= left, need, left


def _guarded(run, cell):
    "A run refused by the cap returns no rows (its cell is then reported as refused)."
    try:
        return run(cell)
    except BudgetExceeded:
        return []


def runWorkload(grids, spec, workers, logDir=None, run=runAnyCell):
    """Run named grids ([(name, cells)], priority order): one pool over all cells, longest first, unless the spec has a
    ledger and the workload check fails (or spec['pooled'] is False), then grid by grid, stopping at the first grid with
    a refused run. Rows come back grouped by grid in the given order, each sorted by row key, whatever the mode."""
    cells = [c for _, g in grids for c in g]
    llm = [c for c in cells if c['runner'] == 'llm']
    budget = Budget(spec['cap'], spec['ledger']) if spec.get('ledger') else None
    spec = dict(spec, floor=workers * max(map(_maxCall, llm)) if budget is not None and llm else 0.0)
    check = checkWorkload(cells, budget, workers) if budget is not None else (True, 0.0, float('inf'))
    pooled = spec.pop('pooled', True) and check[0]
    f = partial(_guarded, run)
    if pooled:
        out = runGrid(withClient(orderCells(cells), spec, logDir), workers, run=f, chunksize=1)
        byGrid = {cellKey(c): name for name, g in grids for c in g}
        parts = {name: [] for name, _ in grids}
        for r in out:
            parts[byGrid[_rowCell(r)]].append(r)
    else:
        parts = {}
        for name, g in grids:
            parts[name] = runGrid(withClient(g, spec, logDir), workers, run=f)
            if any(cellKey(c) not in {_rowCell(r) for r in parts[name]} for c in g if c['runner'] == 'llm'):
                break
    got = {_rowCell(r) for rows in parts.values() for r in rows}
    refused = sorted('|'.join(map(str, cellKey(c))) for c in llm if cellKey(c) not in got)
    done = [n for n, g in grids if n in parts and all(cellKey(c) in got for c in g if c['runner'] == 'llm')]
    rows = [r for name, _ in grids for r in sorted(parts.get(name, []), key=rowKey)]
    return dict(rows=rows, done=done, refused=refused, stopped=f'cap: {len(refused)} runs refused' if refused else None,
                pooled=pooled, need=check[1], left=check[2], floor=spec['floor'])


def mergeTiming(logDir, out):
    "Concatenate the per-run timing files (sorted by name) into `out`; returns the records."
    files = sorted((Path(logDir) / 'timing').glob('*.jsonl')) if logDir else []
    text = ''.join(p.read_text() for p in files)
    Path(out).write_text(text)
    return [json.loads(x) for x in text.splitlines()]


def timingSummary(records, wall):
    "Latency per (model, task) over live calls, calls per second, retries, and wall time per run (`kind='run'` records)."
    runs = {r['key']: r['seconds'] for r in records if r.get('kind') == 'run'}
    records = [r for r in records if r.get('kind', 'call') == 'call']
    out = {}
    for r in records:
        out.setdefault(f"{r['model']}|{r['task']}", []).append(r)
    per = {k: dict(calls=len(v), live=sum(not x['cached'] for x in v),
                   meanSeconds=float(np.mean([x['seconds'] for x in v if not x['cached']] or [np.nan])),
                   p90Seconds=float(np.percentile([x['seconds'] for x in v if not x['cached']] or [np.nan], 90)))
           for k, v in sorted(out.items())}
    return dict(perModelTask=per, calls=len(records), callsPerSecond=len(records) / wall if wall > 0 else float('nan'),
                retries=int(sum(r['retries'] for r in records)), wall=wall, runWall=runs,
                runWallMax=max(runs.values(), default=float('nan')))

## Tests

In [ ]:
import json, tempfile
from pathlib import Path
from explRL.llm_sched import *
from explRL.bench_grid import writeCSV
from explRL.llm_budget import Budget
from explRL.llm_grid import mainGrid, quickGrid, GRIDS, numpyGrid, llmCell

q = quickGrid()
grids = [(s, [c for c in q if c['study'] == s]) for s in ('main', 'sonnet', 'budget', 'liar', 'numpy')]
# sequential calls and the longest-first order: an 8-agent board run before a solo foraging run, numpy last
big = llmCell('main', 'main', 'board', 8, 500, task='board', channel='none')
small = llmCell('main', 'main', 'forage', 1, 500, task='forage', channel='none')
assert seqCalls(big) == 400 and seqCalls(small) == 51 and seqCalls(numpyGrid('board')[0]) == 0
o = orderCells([small, numpyGrid('board')[0], big])
assert o[0] is big and o[-1]['runner'] == 'numpy'
assert orderCells([small, big], latency={('claude-haiku-5-5', 'forage'): 100.0, ('claude-haiku-5-5', 'board'): 1.0})[0] is small

# AC-1: identical outputs for 1, 8 and 32 processes, ordered (pooled) and grid by grid
def csvOf(res):
    p = Path(tempfile.mkdtemp()) / 'r.csv'; writeCSV(res['rows'], p); return p.read_bytes()
ref = None
for w in (1, 8, 32):
    for pooled in (True, False):
        res = runWorkload(grids, dict(kind='fake', pooled=pooled), w)
        b = csvOf(res); ref = ref or b
        assert b == ref and res['done'] == [g for g, _ in grids] and not res['refused']
# rows come back grouped by grid in priority order
studies = [r['study'] for r in res['rows']]
assert studies == sorted(studies, key=[g for g, _ in grids].index)

# AC-3: a workload that does not fit the cap runs grid by grid; refused runs are recorded; spend never passes the cap
tmp = Path(tempfile.mkdtemp())
spec = dict(kind='fake', cap=0.02, ledger=str(tmp / 'spend.jsonl'))
res = runWorkload(grids, spec, 4, run=__import__('explRL.llm_grid', fromlist=['runAnyCell']).runAnyCell)
assert not res['pooled'] and res['refused'] and res['stopped'] and res['floor'] > 0
assert Budget(0.02, tmp / 'spend.jsonl').spent() <= 0.02
assert res['done'] == [g for g, _ in grids][:len(res['done'])]                   # priority order kept
fits, need, left = checkWorkload([c for _, g in grids for c in g], Budget(1000.0, tmp / 'big.jsonl'), 8)
assert fits and need > 0 and left == 1000.0

# AC-4: per-run timing files, merged and summarised
logs = Path(tempfile.mkdtemp())
res = runWorkload(grids[:1], dict(kind='fake'), 4, logDir=str(logs))
recs = mergeTiming(logs, logs / 'timing.jsonl')
calls = sum(int(r['calls']) for r in res['rows'])
callRecs = [r for r in recs if r.get('kind', 'call') == 'call']
assert len(callRecs) == calls and set(callRecs[0]) == {'key', 'model', 'task', 'seconds', 'retries', 'cached'}
runRecs = [r for r in recs if r.get('kind') == 'run']
assert len(runRecs) == sum(c['runner'] == 'llm' for c in grids[0][1]) and sum(r['calls'] for r in runRecs) == calls
s = timingSummary(recs, 2.0)
assert s['calls'] == calls and s['retries'] == 0 and s['callsPerSecond'] == calls / 2.0 and 'claude-haiku-5-5|forage' in s['perModelTask']
assert len(s['runWall']) == len(runRecs) and s['runWallMax'] >= 0

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()